# Proyecto 5 · Entrenamiento del detector de fuego y humo (CNN desde cero)

Este notebook entrena una **red convolucional tipo YOLO** construida y entrenada **desde cero** (sin pesos preentrenados) sobre el dataset **D-Fire** (~21 000 imágenes anotadas con cajas de *humo* y *fuego*).

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4`.

Pasos:
1. Montar Google Drive y cargar el código del proyecto.
2. Descargar / descomprimir D-Fire.
3. **Limpiar el dataset** (imágenes corruptas, etiquetas defectuosas, duplicados, fuga de información, balance) y guardarlo en Drive.
4. Verificar etiquetas, ver estadísticas y calcular las cajas ancla.
5. Entrenar (se guarda cada época en Drive; si Colab se desconecta, vuelve a ejecutar y continúa).
6. Exportar, elegir los umbrales de confianza en validación, evaluar en prueba (mAP, matriz de confusión) y ver predicciones.
7. Descargar `detector_fuego_humo.keras` y `detector_config.json` a la carpeta `modelos/` del proyecto.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import tensorflow as tf
print('TensorFlow', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))

## 1. Google Drive y código del proyecto
Sube la carpeta completa `deteccion_incendios` a tu Drive, por ejemplo en `MyDrive/deteccion_incendios`. Ajusta `RUTA_PROYECTO` si la pusiste en otro lugar.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
RUTA_PROYECTO = '/content/drive/MyDrive/deteccion_incendios'
DIR_ENTRENAMIENTO = RUTA_PROYECTO + '/entrenamiento'   # dataset limpio y anclas (se comparten)
# Cada experimento guarda sus checkpoints e historial en su propia subcarpeta, para no
# mezclarlos con un entrenamiento anterior (el primero quedó directamente en entrenamiento/).
DIR_EXPERIMENTO = DIR_ENTRENAMIENTO + '/v2_zona_ignorar'
DIR_MODELOS = RUTA_PROYECTO + '/modelos'               # modelo final para la app
ZIP_LIMPIO = DIR_ENTRENAMIENTO + '/dfire_limpio.zip'   # dataset ya limpio (se crea en el paso 3)
sys.path.insert(0, RUTA_PROYECTO)

from src import entrenamiento as ent
from src import limpieza
from src import config
CLASES = config.CLASES   # ['humo', 'fuego'] -> D-Fire: 0 = smoke, 1 = fire

## 2. Dataset D-Fire
**Opción A (recomendada):** descarga D-Fire desde el repositorio oficial https://github.com/gaiasd/DFireDataset (enlaces de descarga en el README), sube el `.zip` a tu Drive y pon su ruta en `ZIP_DFIRE`.

**Opción B:** si encuentras una copia en Kaggle, usa la línea comentada con `kagglehub`.

Si el dataset limpio ya existe en Drive (porque ya hiciste el paso 3 antes), esta celda no hace nada: no hace falta volver a copiar el zip original.

In [ ]:
ZIP_DFIRE = '/content/drive/MyDrive/datasets/D-Fire.zip'   # <-- ajusta la ruta
DIR_DATOS = '/content/dfire'

if os.path.exists(ZIP_LIMPIO):
    print('Ya existe el dataset limpio en Drive: se usará directamente.')
elif not os.path.exists(DIR_DATOS):
    !cp "{ZIP_DFIRE}" /content/dfire.zip
    !unzip -q /content/dfire.zip -d {DIR_DATOS}
    !ls {DIR_DATOS}

# Opción B (Kaggle):
# import kagglehub
# DIR_DATOS = kagglehub.dataset_download('<usuario>/<dataset-dfire>')

## 3. Limpieza del dataset
Se ejecuta **una sola vez** (~5–15 min). El resultado se guarda en Drive (`entrenamiento/dfire_limpio.zip`) y las siguientes veces solo se descomprime (~1 min).

| Paso | Qué hace | Por qué |
|---|---|---|
| Imágenes | Elimina corruptas, diminutas, con aspecto extremo o casi uniformes | Una imagen corrupta detiene el entrenamiento |
| Etiquetas | Elimina líneas mal formadas, clases inválidas, cajas diminutas o repetidas; recorta cajas que se salen | La red aprendería ruido |
| Duplicados | Elimina imágenes casi idénticas (hash perceptual) | Fotogramas repetidos de video no aportan |
| Fuga | Elimina de entrenamiento las imágenes casi iguales a una de prueba | Si no, el mAP sale inflado |
| División | Separa train/val por **grupos** de imágenes similares | La validación mide generalización real |
| Balance | Máximo 30 % de imágenes sin fuego ni humo en entrenamiento | Épocas más cortas sin perder ejemplos negativos |
| Rendimiento | Re-codifica todo a 384×384 una sola vez | La GPU no espera a la CPU |

> Para repetir la limpieza con otros umbrales, borra `entrenamiento/dfire_limpio.zip` en Drive y ejecuta `!rm -rf /content/dfire_limpio`.

In [ ]:
DIR_LIMPIO = '/content/dfire_limpio'

if os.path.exists(ZIP_LIMPIO):
    datos = limpieza.restaurar(ZIP_LIMPIO, DIR_LIMPIO)
else:
    datos_crudos = ent.buscar_dataset(DIR_DATOS)
    print('Dataset ORIGINAL:')
    display(ent.resumen_dataset(datos_crudos, CLASES))
    datos, _ = limpieza.limpiar_dataset(datos_crudos, CLASES, DIR_LIMPIO)
    limpieza.empaquetar(DIR_LIMPIO, ZIP_LIMPIO)

print('Reporte de limpieza (cópialo al informe):')
display(limpieza.cargar_reporte(DIR_LIMPIO))
print('Dataset LIMPIO:')
display(ent.resumen_dataset(datos, CLASES))

### Verificar los umbrales de similitud
- **Duplicados:** cada par debe ser prácticamente la misma imagen.
- **Fuga:** cada par debe ser casi la misma escena (por ejemplo, fotogramas seguidos de un video).

Si ves pares que **no** se parecen, los umbrales están demasiado altos: baja `UMBRAL_HASH_DUPLICADO` / `UMBRAL_HASH_SIMILAR` en `src/config.py` y repite la limpieza.

In [ ]:
limpieza.mostrar_imagen_reporte(DIR_LIMPIO + '/ejemplos_duplicados.jpg')
limpieza.mostrar_imagen_reporte(DIR_LIMPIO + '/ejemplos_fuga.jpg')

## 4. Verificación visual, estadísticas y anclas
Las cajas marcadas **fuego** deben estar sobre llamas y las de **humo** sobre humo. Si aparecen invertidas, cambia el orden de `CLASES` en `src/config.py`.

*(Las imágenes se ven cuadradas porque se guardaron a 384×384; la red las ve igual.)*

In [ ]:
ent.mostrar_ejemplos(datos['train'], CLASES, n=6)

In [ ]:
limpieza.graficar_estadisticas(datos, CLASES)

Las anclas son los tamaños "típicos" de las cajas del dataset (k-means). La red predice cada caja como un ajuste sobre el ancla más parecida.

In [ ]:
ANCLAS = ent.calcular_anclas(datos['train'], k=config.NUM_ANCLAS)
print('Anclas (ancho, alto) normalizadas:\n', ANCLAS)

## 5. Modelo y entrenamiento
Se usa **precisión mixta** (float16) para que la GPU entrene casi el doble de rápido. Hiperparámetros:
- `TAM = 320` → cuadrícula de 20×20 celdas
- `LOTE = 16` (si sale error de memoria, bájalo a 8)
- `EPOCAS = 60` con calentamiento + tasa de aprendizaje coseno y parada temprana

Al final de cada época se mide la **detección real en validación** (`val_mAP50`, `val_AP50_humo`, `val_recall_imagen_humo`, ...). El mejor modelo y la parada temprana usan `val_mAP50`, no `val_loss`. Las columnas `perd_caja`, `perd_obj`, `perd_noobj` y `perd_clase` son los términos de la pérdida por separado (sumados dan `loss`).

In [ ]:
from tensorflow import keras
import numpy as np
keras.mixed_precision.set_global_policy('mixed_float16')

TAM, LOTE, EPOCAS = config.TAM_IMAGEN, 16, 60
NUM_CLASES = len(CLASES)

# Al reanudar se reutilizan las mismas anclas con las que se empezó a entrenar
os.makedirs(DIR_ENTRENAMIENTO, exist_ok=True)
ruta_anclas = DIR_ENTRENAMIENTO + '/anclas.npy'
if os.path.exists(ruta_anclas):
    ANCLAS = np.load(ruta_anclas)
    print('Usando anclas guardadas:\n', ANCLAS)
else:
    np.save(ruta_anclas, ANCLAS)

ds_train = ent.crear_dataset(datos['train'], ANCLAS, TAM, NUM_CLASES, LOTE, aumentar=True, barajar=True)
ds_val = ent.crear_dataset(datos['val'], ANCLAS, TAM, NUM_CLASES, LOTE)

from src.modelo import construir_detector
construir_detector(TAM, len(ANCLAS), NUM_CLASES).summary()

In [ ]:
# Si Colab se desconecta: ejecuta de nuevo TODAS las celdas desde el inicio;
# el entrenamiento continúa desde la última época guardada en Drive.
# (Para empezar de cero, borra ultimo.keras, mejor.keras e historial.csv de la carpeta
#  DIR_EXPERIMENTO, o cambia su nombre en la sección 1. NO borres dfire_limpio.zip.)
# Si el entrenamiento ya terminó y solo quieres repetir la sección 6 (umbrales y
# evaluación), SÁLTATE esta celda: si se detuvo antes de la última época por la parada
# temprana, al ejecutarla seguiría entrenando.
ruta_mejor = ent.entrenar(ds_train, ds_val, ANCLAS, NUM_CLASES, TAM, DIR_EXPERIMENTO, epocas=EPOCAS,
                          split_val=datos['val'], clases=CLASES,
                          umbral=config.umbrales_por_clase(CLASES))

In [ ]:
ent.graficar_historial(DIR_EXPERIMENTO + '/historial.csv')

## 6. Exportar, elegir umbrales y evaluar
Se exporta el mejor modelo en float32 (para correr rápido en CPU en la app). Con **ese mismo archivo** se hacen dos cosas, en este orden:
1. **Elegir los umbrales de confianza** mirando el conjunto de **validación** (6.1).
2. **Evaluar** en el conjunto de **prueba** con esos umbrales (6.2).

El orden importa: si los umbrales se eligieran mirando el conjunto de prueba, sus métricas dejarían de ser una medida limpia.

In [ ]:
modelo = ent.exportar_modelo(DIR_EXPERIMENTO + '/mejor.keras', ANCLAS, CLASES, TAM, DIR_MODELOS)

### 6.1 Elegir los umbrales de confianza (en validación)
Una imagen "tiene humo" si alguna caja de humo supera el umbral de humo (igual con el fuego). Para cada umbral, la gráfica y la tabla muestran:
- **recall**: de las imágenes que tienen la clase, en cuántas se detecta. Si baja, se pierden más incendios.
- **precisión**: de las imágenes donde el modelo avisa, en cuántas es cierto.
- **falsas alarmas en imágenes normales**: fracción de las imágenes sin fuego ni humo donde avisa.

Bajar el umbral sube el recall y también las falsas alarmas; no hay un valor "correcto", es una decisión. Como en este proyecto es peor **no ver el humo** que dar una falsa alarma, busca el umbral de humo más bajo cuyas falsas alarmas todavía te parezcan aceptables. El umbral de una clase no cambia las cifras de la otra. La línea punteada es el umbral actual de `src/config.py`.

In [ ]:
UMBRALES = config.umbrales_por_clase(CLASES)   # los de src/config.py, en el orden de CLASES
barrido, preds_val = ent.explorar_umbrales(modelo, datos['val'], ANCLAS, CLASES, TAM, actuales=UMBRALES)

### 6.2 Evaluar en el conjunto de prueba
El conjunto de prueba no tiene fuga con el entrenamiento y no se usó para elegir nada:
- **AP50 / mAP50**: calidad de las cajas (una caja es correcta si se solapa ≥ 50 % con la real). No dependen del umbral.
- **Matriz de eventos por imagen**: normal / solo humo / solo fuego / fuego y humo.
- **Precisión y recall por imagen** para fuego y humo, con los umbrales elegidos.

Cuando te decidas por unos umbrales, escríbelos también en `UMBRALES_CLASE` de `src/config.py` (en Drive y en tu computador) para que la app use los mismos.

In [ ]:
# Para probar otros umbrales sin editar src/config.py, quita el # de la línea siguiente
# y cambia los números (mismo orden que CLASES: humo, fuego):
# UMBRALES = [0.15, 0.35]
resultados, preds_test = ent.evaluar(modelo, datos['test'], ANCLAS, CLASES, TAM, umbral=UMBRALES)

In [ ]:
ent.mostrar_predicciones(preds_test, datos['test'], ANCLAS, CLASES, umbral=UMBRALES)

In [ ]:
# Guarda las métricas, el barrido de umbrales y el reporte de limpieza para el informe
import json, shutil
with open(DIR_MODELOS + '/metricas_test.json', 'w') as f:
    json.dump(resultados, f, indent=2)
barrido.to_csv(DIR_MODELOS + '/barrido_umbrales_val.csv', index=False)
for archivo in ['reporte_limpieza.csv', 'ejemplos_duplicados.jpg', 'ejemplos_fuga.jpg']:
    if os.path.exists(DIR_LIMPIO + '/' + archivo):
        shutil.copy(DIR_LIMPIO + '/' + archivo, DIR_MODELOS)
print('Umbrales usados:', dict(zip(CLASES, UMBRALES)))
print(json.dumps(resultados, indent=2))

## 7. Llevar el modelo a tu computador
En Drive, dentro de `deteccion_incendios/modelos/`, quedaron:
- `detector_fuego_humo.keras` y `detector_config.json` → **necesarios para la app**
- `metricas_test.json`, `barrido_umbrales_val.csv`, `reporte_limpieza.csv`, `ejemplos_duplicados.jpg`, `ejemplos_fuga.jpg` → para el informe

Descárgalos a la carpeta `modelos/` del proyecto local y ejecuta la app con `streamlit run app.py`.